# 02k — InceptionV3 + CBAM Training Pipeline with Synthetic Data

**Project:** UREP 32-0210-250078 | Crack Classification

**Architecture:** InceptionV3 backbone + CBAM (Convolutional Block Attention Module)

**Experiment:** Same architecture, hyperparameters, and seeds as `02b_training_cbam.ipynb`.
The only difference is that 806 synthetic AutoCAD-generated crack images are
added to the training set (102 debonding/corrosion, 302 flexural, 402 shear).
Val/test sets are identical to the baseline for fair comparison.

Same 3-stage freeze/unfreeze protocol as base InceptionV3.
CBAM layers are always trainable.

**Reference:** Woo et al. (2018), ECCV.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import torch
import torch.nn as nn

import config
from src.dataset import get_dataloaders, compute_class_weights, prepare_synthetic_split
from src.model_cbam import InceptionV3CBAM
from src.model import freeze_backbone, unfreeze_from, unfreeze_all
from src.trainer import train_model
from src.evaluation import plot_training_history, evaluate_model
from src.device import print_device_summary, get_device, set_seed

# Reproducibility
set_seed(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "cbam_synthetic")
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)

device_config = print_device_summary()
device = get_device()
STAGE_BATCH = device_config["batch_sizes"]
NUM_WORKERS = device_config["num_workers"]

# Materialize synthetic split
SPLIT = config.SPLIT_SYNTHETIC_DIR
prepare_synthetic_split()

print(f"\nModel: InceptionV3 + CBAM (with Synthetic Data)")
print(f"Device: {device}")
print(f"Split: {SPLIT}")

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    split_dir=SPLIT,
    batch_size=STAGE_BATCH[1], img_size=config.IMG_SIZE,
    normalize="imagenet", num_workers=NUM_WORKERS,
)

class_weight_dict = compute_class_weights(split_dir=SPLIT)
class_weight_tensor = torch.tensor(
    [class_weight_dict[i] for i in range(config.NUM_CLASSES)], dtype=torch.float32,
).to(device)

In [ ]:
model = InceptionV3CBAM().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## Stage 1 — Feature Extraction

In [ ]:
freeze_backbone(model)
criterion = nn.CrossEntropyLoss(weight=class_weight_tensor)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=config.STAGE1_LR,
)

history1 = train_model(
    model, train_loader, val_loader, optimizer, criterion, device,
    epochs=config.STAGE1_EPOCHS, output_dir=OUTPUT_DIR, stage=1,
    model_name="cbam_synth",
)

## Stage 2 — Partial Fine-Tuning

In [ ]:
if STAGE_BATCH[2] != STAGE_BATCH[1]:
    train_loader, val_loader, _ = get_dataloaders(
        split_dir=SPLIT,
        batch_size=STAGE_BATCH[2], img_size=config.IMG_SIZE,
        normalize="imagenet", num_workers=NUM_WORKERS,
    )

unfreeze_from(model, "Mixed_7a")
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=config.STAGE2_LR,
)

history2 = train_model(
    model, train_loader, val_loader, optimizer, criterion, device,
    epochs=config.STAGE2_EPOCHS, output_dir=OUTPUT_DIR, stage=2,
    model_name="cbam_synth",
)

## Stage 3 — Full Fine-Tuning

In [ ]:
if STAGE_BATCH[3] != STAGE_BATCH[2]:
    train_loader, val_loader, _ = get_dataloaders(
        split_dir=SPLIT,
        batch_size=STAGE_BATCH[3], img_size=config.IMG_SIZE,
        normalize="imagenet", num_workers=NUM_WORKERS,
    )

unfreeze_all(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.STAGE3_LR)

history3 = train_model(
    model, train_loader, val_loader, optimizer, criterion, device,
    epochs=config.STAGE3_EPOCHS, output_dir=OUTPUT_DIR, stage=3,
    model_name="cbam_synth",
)

## Training Curves & Evaluation

In [ ]:
plot_training_history(
    [history1, history2, history3],
    output_dir=OUTPUT_DIR,
    stage_names=["Feature Extraction", "Partial Fine-Tuning", "Full Fine-Tuning"],
    model_name="cbam_synth",
)

In [ ]:
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "models", "best_model.pt"))

# Test from ORIGINAL split (no synthetic) for fair comparison
_, _, test_loader = get_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[1], img_size=config.IMG_SIZE,
    normalize="imagenet", num_workers=NUM_WORKERS,
)
metrics = evaluate_model(model, test_loader, device, output_dir=OUTPUT_DIR, model_name="cbam_synth")